In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

file_path = "/Volumes/workspace/default/assignment_data/Sample - Superstore.csv"

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

display(
    df.select(
        "Row ID",
        "Order ID",
        "Order Date",
        "Customer Name",
        "Category",
        "Sales",
        "Profit"
    ).limit(10)
)

Row ID,Order ID,Order Date,Customer Name,Category,Sales,Profit
1,CA-2016-152156,2016-11-08,Claire Gute,Furniture,261.96,41.9136
2,CA-2016-152156,2016-11-08,Claire Gute,Furniture,731.94,219.582
3,CA-2016-138688,2016-06-12,Darrin Van Huff,Office Supplies,14.62,6.8714
4,US-2015-108966,2015-10-11,Sean O'Donnell,Furniture,957.5775,-383.031
5,US-2015-108966,2015-10-11,Sean O'Donnell,Office Supplies,22.368,2.5164
6,CA-2014-115812,2014-06-09,Brosina Hoffman,Furniture,48.86,14.1694
7,CA-2014-115812,2014-06-09,Brosina Hoffman,Office Supplies,7.28,1.9656
8,CA-2014-115812,2014-06-09,Brosina Hoffman,Technology,907.152,90.7152
9,CA-2014-115812,2014-06-09,Brosina Hoffman,Office Supplies,18.504,5.7825
10,CA-2014-115812,2014-06-09,Brosina Hoffman,Office Supplies,114.9,34.47


In [0]:
df = df.toDF(*[c.replace(" ", "_") for c in df.columns])

print(df.columns)

['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub-Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [0]:
delta_path = "/Volumes/workspace/default/assignment_data/superstore_delta"

df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

delta_df = spark.read.format("delta").load(delta_path)

display(
    delta_df.select(
        "Row_ID",
        "Order_ID",
        "Order_Date",
        "Customer_Name",
        "Category",
        "Sales",
        "Profit"
    ).limit(10)
)

Row_ID,Order_ID,Order_Date,Customer_Name,Category,Sales,Profit
1,CA-2016-152156,2016-11-08,Claire Gute,Furniture,261.96,41.9136
2,CA-2016-152156,2016-11-08,Claire Gute,Furniture,731.94,219.582
3,CA-2016-138688,2016-06-12,Darrin Van Huff,Office Supplies,14.62,6.8714
4,US-2015-108966,2015-10-11,Sean O'Donnell,Furniture,957.5775,-383.031
5,US-2015-108966,2015-10-11,Sean O'Donnell,Office Supplies,22.368,2.5164
6,CA-2014-115812,2014-06-09,Brosina Hoffman,Furniture,48.86,14.1694
7,CA-2014-115812,2014-06-09,Brosina Hoffman,Office Supplies,7.28,1.9656
8,CA-2014-115812,2014-06-09,Brosina Hoffman,Technology,907.152,90.7152
9,CA-2014-115812,2014-06-09,Brosina Hoffman,Office Supplies,18.504,5.7825
10,CA-2014-115812,2014-06-09,Brosina Hoffman,Office Supplies,114.9,34.47


In [0]:
delta_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in delta_df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row_ID|Order_ID|Order_Date|Ship_Date|Ship_Mode|Customer_ID|Customer_Name|Segment|Country|City|State|Postal_Code|Region|Product_ID|Category|Sub-Category|Product_Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [0]:
clean_df = delta_df.dropDuplicates()

display(
    clean_df.select(
        "Row_ID",
        "Order_ID",
        "Customer_Name",
        "Category",
        "Sales",
        "Profit"
    ).limit(10)
)

Row_ID,Order_ID,Customer_Name,Category,Sales,Profit
5,US-2015-108966,Sean O'Donnell,Office Supplies,22.368,2.5164
17,CA-2014-105893,Pete Kriz,Office Supplies,665.88,13.3176
21,CA-2014-143336,Zuschuss Donatelli,Office Supplies,22.72,7.384
53,CA-2015-115742,Darren Powers,Furniture,89.99,17.0981
118,CA-2015-110457,Dave Kipp,Furniture,787.53,165.3813
140,CA-2016-145583,Lena Creighton,Furniture,43.12,20.6976
145,CA-2017-155376,Sandra Glassco,Office Supplies,839.43,218.2518
147,CA-2014-110072,Maureen Gastineau,Furniture,93.888,12.9096
186,CA-2016-105018,Sally Knutson,Office Supplies,7.16,3.4368
200,US-2017-124303,Fred Hopkins,Office Supplies,16.056,5.8203


In [0]:
unique_order = (
    clean_df.groupBy("Order_ID")
    .count()
    .filter(col("count") == 1)
    .select("Order_ID")
    .first()[0]
)

updated_record = (
    clean_df.filter(col("Order_ID") == unique_order)
    .withColumn("Sales", lit(9999.99))
)

new_record = (
    clean_df.filter(col("Order_ID") == unique_order)
    .withColumn("Order_ID", lit("NEW-ORDER-001"))
    .withColumn("Row_ID", lit(99999))
    .withColumn("Sales", lit(5000.00))
)

incremental_df = updated_record.union(new_record)

display(
    incremental_df.select(
        "Row_ID",
        "Order_ID",
        "Customer_Name",
        "Category",
        "Sales",
        "Profit"
    )
)

Row_ID,Order_ID,Customer_Name,Category,Sales,Profit
17,CA-2014-105893,Pete Kriz,Office Supplies,9999.99,13.3176
99999,NEW-ORDER-001,Pete Kriz,Office Supplies,5000.0,13.3176


In [0]:
delta_table = DeltaTable.forPath(spark, delta_path)

(delta_table.alias("target")
 .merge(
     incremental_df.alias("source"),
     "target.Order_ID = source.Order_ID"
 )
 .whenMatchedUpdate(set={
     "Sales": "source.Sales"
 })
 .whenNotMatchedInsertAll()
 .execute())

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
merged_df = spark.read.format("delta").load(delta_path)

display(
    merged_df.filter(
        col("Order_ID").isin(unique_order, "NEW-ORDER-001")
    ).select(
        "Row_ID",
        "Order_ID",
        "Customer_Name",
        "Category",
        "Sales",
        "Profit"
    )
)

Row_ID,Order_ID,Customer_Name,Category,Sales,Profit
17,CA-2014-105893,Pete Kriz,Office Supplies,9999.99,13.3176
99999,NEW-ORDER-001,Pete Kriz,Office Supplies,5000.0,13.3176


In [0]:
print("Total Rows:", merged_df.count())

Total Rows: 9995


In [0]:
merged_df.groupBy("Row_ID") \
    .count() \
    .filter(col("count") > 1) \
    .show()

+------+-----+
|Row_ID|count|
+------+-----+
+------+-----+



In [0]:
display(
    merged_df.select(
        "Row_ID",
        "Order_ID",
        "Order_Date",
        "Customer_Name",
        "Category",
        "Sales",
        "Profit"
    ).limit(15)
)

Row_ID,Order_ID,Order_Date,Customer_Name,Category,Sales,Profit
1,CA-2016-152156,2016-11-08,Claire Gute,Furniture,261.96,41.9136
2,CA-2016-152156,2016-11-08,Claire Gute,Furniture,731.94,219.582
3,CA-2016-138688,2016-06-12,Darrin Van Huff,Office Supplies,14.62,6.8714
4,US-2015-108966,2015-10-11,Sean O'Donnell,Furniture,957.5775,-383.031
5,US-2015-108966,2015-10-11,Sean O'Donnell,Office Supplies,22.368,2.5164
6,CA-2014-115812,2014-06-09,Brosina Hoffman,Furniture,48.86,14.1694
7,CA-2014-115812,2014-06-09,Brosina Hoffman,Office Supplies,7.28,1.9656
8,CA-2014-115812,2014-06-09,Brosina Hoffman,Technology,907.152,90.7152
9,CA-2014-115812,2014-06-09,Brosina Hoffman,Office Supplies,18.504,5.7825
10,CA-2014-115812,2014-06-09,Brosina Hoffman,Office Supplies,114.9,34.47


# Assignment Summary

- Successfully loaded the Superstore dataset into a Delta table using PySpark.
- Renamed all column names by replacing spaces with underscores for easier processing.
- Performed data quality checks and found only **2 null values** in the dataset.
- Removed duplicate records using the `dropDuplicates()` function.
- Created an incremental dataset containing **one updated record** and **one new record**.
- Applied the Delta Lake **MERGE** operation to update the existing record and insert the new record.
- Validated the final dataset by checking the row count and confirming that there were **no duplicate `Row_ID` values** after the merge.
- Successfully displayed the final Delta table, confirming that all changes were applied correctly.